In [1]:
import chess.svg
import numpy as np
from pathlib import Path
import torch
import pytorch_lightning as pl

from chess_gnn.models import *
from chess_gnn.utils import PGNBoardHelper, ChessPoint
from chess_gnn.tokenizers import SimpleChessTokenizer
from chess_gnn.visualization import MaskVisualizationHelper

import dash
from dash import dcc, html, Input, Output, State
import plotly.graph_objects as go
import chess
import chess.svg
import base64

In [2]:
class ChessBoardArray:
    def __init__(self, board_array: np.array):
        self.board_array = board_array
        if len(self.board_array) != 64:
            raise ValueError("Board array must be of length 64")
        self.tokenizer = SimpleChessTokenizer()
    
    def __len__(self):
        return len(self.board_array)
    
    def __getitem__(self, item):
        return self.board_array[item]
    
    def to_pieces(self):
        untokenized = self.tokenizer.untokenize(self.board_array)
        untokenized = [token if token != '.' else None for token in untokenized]
        return untokenized

In [3]:
def board_to_svg_image(board: ChessBoardArray):
    images = []
    pieces = board.to_pieces()
    for idx in range(64):
        piece = pieces[idx]
        point = ChessPoint.from_1d(idx)
        if piece:
            svg = chess.svg.piece(chess.Piece.from_symbol(piece))
            svg_bytes = svg.encode('utf-8')
            uri = f"data:image/svg+xml;base64,{base64.b64encode(svg_bytes).decode('utf-8')}"
            images.append(dict(
                source=uri,
                xref="x", yref="y",
                x=point.x,
                y=7-point.y,
                sizex=1.0, sizey=1.0,
                xanchor="center", yanchor="middle",
                layer="above"
            ))
    return images

def create_board_figure(board: ChessBoardArray, mask: np.array):
    # Create base array for heatmap
    z = np.flipud(np.reshape(mask, (8,8)))

    # Create transparent red heatmap as mask
    heatmap = go.Heatmap(
        z=z,
        x=list(range(8)),
        y=list(range(8)),
        colorscale=[[0, 'rgba(0,0,0,0)'], [1, 'rgba(255,0,0,0.4)']],
        zmin=0,
        zmax=1,
        showscale=False,
        hoverinfo='skip',
        xgap=0,
        ygap=0,
        opacity=1.0,
    )

    fig = go.Figure(data=[heatmap])

    fig.update_layout(
        width=320,
        height=320,
        margin=dict(l=0, r=0, t=0, b=0),
        yaxis=dict(
            scaleanchor="x",
            scaleratio=1,
            showgrid=False,
            zeroline=False,
            showticklabels=False,
            range=[-0.5, 7.5],
            fixedrange=True
        ),
        xaxis=dict(
            constrain='domain',
            showgrid=False,
            zeroline=False,
            showticklabels=False,
            range=[-0.5, 7.5],
            fixedrange=True
        ),
        images=board_to_svg_image(board),
        shapes=[],
        plot_bgcolor="white",
        paper_bgcolor="white",
    )

    return fig

def create_dual_board_app(boards1: list[ChessBoardArray], 
                          boards2: list[ChessBoardArray], 
                          # boards3: list[ChessBoardArray],
                          # boards4: list[ChessBoardArray],
                          masks):
    assert len(boards1) == len(boards2), "Board lists must be the same length"

    num_steps = len(boards1)
    app = dash.Dash(__name__)

    app.layout = html.Div([
        html.Div([
            html.Button("Prev", id="prev-btn", n_clicks=0),
            html.Button("Next", id="next-btn", n_clicks=0),
            html.Span(id="step-label", style={"marginLeft": "1rem"}),
        ], style={"marginBottom": "1rem"}),

        dcc.Store(id="current-step", data=0),

        html.Div([
            dcc.Graph(id='left-board', config={"displayModeBar": False}),
            dcc.Graph(id='right-board', config={"displayModeBar": False}),
        ], style={"display": "flex", "justifyContent": "center", "gap": "2rem"}),
    ])

    @app.callback(
        Output('current-step', 'data'),
        Output('step-label', 'children'),
        Input('prev-btn', 'n_clicks'),
        Input('next-btn', 'n_clicks'),
        State('current-step', 'data')
    )
    def update_step(prev, nxt, current):
        ctx = dash.callback_context.triggered_id
        if ctx == 'prev-btn':
            current = max(0, current - 1)
        elif ctx == 'next-btn':
            current = min(num_steps - 1, current + 1)
        return current, f"Step: {current} / {num_steps - 1}"

    @app.callback(
        Output('left-board', 'figure'),
        Output('right-board', 'figure'),
        Input('current-step', 'data')
    )
    def update_boards(idx):
        mask = masks[idx]
        return (
            create_board_figure(boards1[idx], mask),
            create_board_figure(boards2[idx], mask),
        )

    return app

In [28]:
ckpt_file = '/Users/ray/models/chess/transformer/dcf5a7aa-1a04-435e-a72d-c7e7fbec9612/epoch=1-step=190000.ckpt'
model = ChessTransformer.load_from_checkpoint(ckpt_file)

model.eval()
pl.seed_everything(42)

Seed set to 42


42

In [33]:
ckpt = torch.load(ckpt_file, map_location='cpu')

# 1. Inspect basic training metadata
print("== Checkpoint metadata ==")
print("epoch:", ckpt.get("epoch"))
print("global_step:", ckpt.get("global_step"))
print("state_dict keys:", list(ckpt['state_dict'].keys())[:5])

# 2. Manually create model and load weights
model_manual = ChessTransformer(**ckpt["hyper_parameters"])
missing, unexpected = model_manual.load_state_dict(ckpt["state_dict"], strict=False)
print("== State dict load ==")
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

untrained_model = ChessTransformer(**ckpt["hyper_parameters"])

# 3. Compare a trained parameter
trained_param = model_manual.embedding_table.data
print("Embedding mean:", trained_param.mean().item(), "std:", trained_param.std().item())

untrained_param = untrained_model.embedding_table.data
print("Untrained model embedding mean:", untrained_param.mean().item(), "std:", untrained_param.std().item())

auto_param = model.embedding_table.data
print("Lightning loaded mean:", auto_param.mean().item(), "std:", auto_param.std().item())

/var/folders/x0/rmp25fy116j6q3pcyk4sbg6w0000gn/T/ipykernel_55126/2779184134.py:1: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.



== Checkpoint metadata ==
epoch: 1
global_step: 190000
state_dict keys: ['current_board_cls_token', 'next_board_cls_token', 'embedding_table', 'whose_move_embedding', 'pos_embedding']
== State dict load ==
Missing keys: []
Unexpected keys: []
Embedding mean: -0.00046209385618567467 std: 0.06632700562477112
Untrained model embedding mean: -0.0008158385171554983 std: 0.04418972507119179
Lightning loaded mean: -0.00046209385618567467 std: 0.06632700562477112


In [7]:
pgn = PGNBoardHelper(Path('/Users/ray/Datasets/chess/Carlsen.pgn'))
for i in range(1):
    pgn.get_game()

tokenizer = SimpleChessTokenizer()
game_batch = pgn.get_game_batch(tokenizer=tokenizer)

In [41]:
from chess_gnn.data import HDF5ChessDataset
from torch.utils.data import DataLoader

file = '/Users/ray/Datasets/chess/Carlsen_transformer/test/data.h5'
dataset = HDF5ChessDataset(str(file), 64, mode='transformer')
dl = DataLoader(dataset, batch_size=1, num_workers=1, shuffle=False)

batch = next(iter(dl))

In [53]:
batch['whose_move']

tensor([[0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
         0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
         0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]])

In [54]:
game_batch['whose_move']

tensor([[0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
         0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]])

In [57]:
loss_model = untrained_model
loss_model.eval()
with torch.no_grad():
    loss = loss_model(batch)

In [58]:
loss

{'current_board_loss': tensor(2.2529),
 'next_board_loss': tensor(2.2475),
 'loss': tensor(4.5004)}

In [50]:
helper = MaskVisualizationHelper(transformer=model)
with torch.no_grad():
    m, current, nxt = helper.get_preds(batch)

In [51]:
current_pred = [ChessBoardArray(arr) for arr in current.numpy()]
current_label = [ChessBoardArray(arr) for arr in batch['board'].squeeze().numpy()]
nxt_pred = [ChessBoardArray(arr) for arr in nxt.numpy()]
nxt_label = [ChessBoardArray(arr) for arr in batch['next_board'].squeeze().numpy()]

In [52]:
app = create_dual_board_app(current_label, current_pred, m)
app.run(mode="jupyter-inline", port=8051)

In [16]:
tokenizer.vocab

['.', 'B', 'K', 'N', 'P', 'Q', 'R', 'b', 'k', 'n', 'p', 'q', 'r']

In [17]:
tokenizer.inverse_vocab

{'.': 0,
 'B': 1,
 'K': 2,
 'N': 3,
 'P': 4,
 'Q': 5,
 'R': 6,
 'b': 7,
 'k': 8,
 'n': 9,
 'p': 10,
 'q': 11,
 'r': 12}